# AI651 PA2 - Question 1 (river proposal figure) and Question 2 (Motion or Memory?)

Driver notebook: all code lives in the public repo https://github.com/RazaTypesCode/DLSTG (branch `PA2`); this notebook clones it and runs it. Nothing uses Google Drive.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

**Order matters.** Q2.5 must be committed to the repo before the stress tests (section 5) and Q2.11 before training B, C, D (section 7). Two gate cells check this and stop otherwise.

**If the runtime is recycled**, `/content` is wiped. Run the *Backup* cell (section 11) after each long stage and download the zip; after a reset run section 1, then the *Restore* cell, then continue (finished runs are skipped, unfinished ones resume).

## 1. Setup

In [ ]:
import os, torch
REPO = 'https://github.com/RazaTypesCode/DLSTG.git'
BRANCH = 'PA2'
if not os.path.exists('/content/DLSTG'):
    !git clone --branch $BRANCH --single-branch $REPO /content/DLSTG
else:
    !git -C /content/DLSTG pull
os.environ['PA2_RUNS'] = '/content/runs'   # checkpoints + weights (local to this runtime)
os.environ['PA2_DATA'] = '/content/data'   # Moving MNIST download
os.makedirs('/content/runs', exist_ok=True)
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY - switch runtime to T4 GPU')
!git -C /content/DLSTG log --oneline -5

## 2. Question 1 - architecture diagram
The written proposal is `Question1/Q1_river_proposal.tex` in the repo; this cell only draws its architecture figure.

In [ ]:
%cd /content/DLSTG/Question1
!python q1_diagram.py
from IPython.display import Image, display
display(Image('figures/q1_architecture.png'))
%cd /content/DLSTG/Question2

## 3. Question 2 - Table 1 (transform / window checks) and Figure 1 (optical flow)
Table 1 must match the *Expected* values before anything else is trusted.

In [ ]:
%cd /content/DLSTG/Question2
!python check_table1.py

In [ ]:
!python fig1_flow.py
from IPython.display import Image, display
display(Image('figures/fig1.png'))

## 4. Train baseline model A (seeds 0 and 1) and run Table 2 / Figure 2

In [ ]:
!python train.py --model A --seed 0

In [ ]:
!python train.py --model A --seed 1

In [ ]:
!python stage_A1.py
from IPython.display import Image, display
display(Image('figures/fig2.png'))

### GATE 1 - Q2.5
Write your Q2.5 prediction in `predictions/Q2.5.md`, **commit and push** it, then run this cell. It stops if the file is not in the pushed repo, and prints its commit time.

In [ ]:
import subprocess
subprocess.run(['git', '-C', '/content/DLSTG', 'pull'], check=True)
p = 'predictions/Q2.5.md'
t = subprocess.run(['git', '-C', '/content/DLSTG', 'log', '-1', '--format=%cI', '--', p], capture_output=True, text=True).stdout.strip()
assert t, 'Q2.5 not found in the pushed repo: commit and push predictions/Q2.5.md first.'
print('Q2.5 committed at', t)

## 5. Stress tests on model A: Tables 3-4, Figures 3-5

In [ ]:
%cd /content/DLSTG/Question2
!python stage_A2.py
!cat results/table3.md results/table4.md
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figures/fig3_*.png')) + ['figures/fig4.png', 'figures/fig5.png']:
    print(f); display(Image(f))

### GATE 2 - Q2.11
Write your Q2.11 prediction (max 3 sentences) in `predictions/Q2.11.md`, **commit and push** it, then run this cell.

In [ ]:
import subprocess
subprocess.run(['git', '-C', '/content/DLSTG', 'pull'], check=True)
p = 'predictions/Q2.11.md'
t = subprocess.run(['git', '-C', '/content/DLSTG', 'log', '-1', '--format=%cI', '--', p], capture_output=True, text=True).stdout.strip()
assert t, 'Q2.11 not found in the pushed repo: commit and push predictions/Q2.11.md first.'
print('Q2.11 committed at', t)

## 6. Models B and C (seeds 0 and 1)

In [ ]:
for m in ('B', 'C'):
    for s in (0, 1):
        !python train.py --model $m --seed $s

## 7. Model D: train both candidates (seed 0), choose on validation only, then seed 1

In [ ]:
!python train.py --model D_p8 --seed 0
!python train.py --model D_dil2 --seed 0

In [ ]:
!python select_D.py
!python train.py --model D --seed 1

## 8. Tables 5-6 and Figure 6 (final model = symmetrised D)

In [ ]:
!python stage_final.py
!cat results/table5.md results/table6.md
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figures/fig6a_*.png')) + ['figures/fig6b.png']:
    print(f); display(Image(f))

## 9. Hardware and timing record (for the report)

In [ ]:
import json, glob
for f in sorted(glob.glob('/content/runs/*/log.json')):
    d = json.load(open(f))
    print(f"{d['model']:7s} seed {d['seed']}  params {d['params']:>8,}  reach {d['reach_px']:>2} px  {d['mean_epoch_sec']:.1f} s/epoch  [{d['hardware']}]")

## 10. Download the outputs (tables, figures, GIFs)
Download `pa2_outputs.zip` and unzip it at the **root of your repo** (it contains `Question1/figures`, `Question2/results`, `Question2/figures`), then commit (the handout needs the GIFs in the public repo).

In [ ]:
%cd /content/DLSTG
!zip -qr /content/pa2_outputs.zip Question1/figures Question2/results Question2/figures
from google.colab import files
files.download('/content/pa2_outputs.zip')
%cd /content/DLSTG/Question2

## 11. Backup / restore of trained models (no Drive)
Run **Backup** after each long training stage and keep the downloaded zip. After a runtime reset: run section 1, then **Restore**, then continue from where you stopped.

In [ ]:
# BACKUP
!cd /content && zip -qr pa2_runs_backup.zip runs
from google.colab import files
files.download('/content/pa2_runs_backup.zip')

In [ ]:
# RESTORE (after a reset and section 1): choose pa2_runs_backup.zip when prompted
%cd /content
from google.colab import files
up = files.upload()
!unzip -qo /content/pa2_runs_backup.zip -d /content
!ls /content/runs
%cd /content/DLSTG/Question2